# AgeLens — 09 Mortality Analysis Authorization and Cohort Definition

This notebook governs the transition from the completed cross-sectional AgeLens V1 replication to linked-mortality analysis.

## Purpose

The notebook:

- verifies that canonical cross-sectional outputs are released;
- verifies that mortality analysis is still unauthorized;
- joins the canonical participant file to the separately ingested NCHS public-use linked mortality file;
- defines the mortality cohort without fitting outcome models;
- creates cycle-specific weighted Phenotypic Age acceleration;
- approves the mortality cohort and model specification as `D-015`;
- creates `Mortality_Analysis_Protocol.md`;
- authorizes notebook 10 to fit mortality models;
- keeps mortality **results** non-reportable until notebook 10 passes its own validation and release gate.

## Governed primary analysis

- Population: canonical harmonized complete cases aged **20 years or older**
- Linkage: `ELIGSTAT == 1`
- Outcome: all-cause mortality, `MORTSTAT`
- Time origin: MEC examination
- Follow-up: `PERMTH_EXM`, expressed in months
- Exposure: cycle-specific survey-weighted residual of canonical Supplement Phenotypic Age on chronological age
- Primary effect: hazard ratio per 5-year higher Phenotypic Age acceleration
- Primary adjusted model: chronological age, sex, race/ethnicity, and NHANES cycle
- Survey design: `WTSAF4YR`, cycle-unique strata, and cycle-unique PSUs
- Estimator planned for notebook 10: R `survey::svycoxph`

## Required sensitivities

- no-topcode cohort;
- Erratum constants;
- creatinine shifts of `+0.11`, `+0.17`, and `+0.23 mg/dL`;
- exclusion of deaths within the first 12 follow-up months.

Cause-specific mortality is not authorized in AgeLens V1.


In [4]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import re
import shutil

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_FOLDER_NAME = "nhanes"
DECISION_ID = "D-015"
AUTHORIZATION_DATE = "2026-07-22"
AUTHORIZATION_MARKER = (
    "<!-- AGE-LENS MORTALITY AUTHORIZATION 2026-07-22 -->"
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Current working directory: {Path.cwd().resolve()}")


numpy: 2.4.6
pandas: 2.3.3
Current working directory: <PROJECT_ROOT>\notebooks


In [5]:
def find_project_root(
    folder_name: str = PROJECT_FOLDER_NAME,
) -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate

    raise FileNotFoundError(
        f"Could not find a parent folder named '{folder_name}'."
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def replace_document_field(
    text: str,
    field: str,
    value: str,
) -> str:
    pattern = re.compile(
        rf"(?m)^(\|\s*{re.escape(field)}\s*\|\s*)"
        rf"([^|]*?)(\s*\|)$"
    )

    updated, count = pattern.subn(
        lambda match: (
            f"{match.group(1)}{value}{match.group(3)}"
        ),
        text,
        count=1,
    )

    if count != 1:
        raise RuntimeError(
            f"Could not update document field: {field}"
        )

    return updated


def extract_document_version(text: str) -> str:
    match = re.search(
        r"(?m)^\|\s*Version\s*\|\s*([^|]+?)\s*\|$",
        text,
    )

    if not match:
        raise RuntimeError(
            "Could not extract document version."
        )

    return match.group(1).strip()


def add_revision_row(
    text: str,
    row: str,
) -> str:
    marker = (
        "*This file is a single authoritative document"
    )
    marker_index = text.find(marker)

    if marker_index < 0:
        raise RuntimeError(
            "Authoritative-document marker was not found."
        )

    before = text[:marker_index].rstrip()
    after = text[marker_index:]

    if row in before:
        return text

    return before + "\n" + row + "\n\n" + after


def dataframe_to_markdown(
    frame: pd.DataFrame,
) -> str:
    columns = [str(column) for column in frame.columns]
    header = "| " + " | ".join(columns) + " |"
    separator = (
        "| "
        + " | ".join(["---"] * len(columns))
        + " |"
    )

    rows = []

    for _, row in frame.iterrows():
        values = []

        for value in row:
            if pd.isna(value):
                text = ""
            elif isinstance(
                value,
                (float, np.floating),
            ):
                text = f"{float(value):.6g}"
            else:
                text = str(value)

            values.append(
                text.replace("|", "\\|")
            )

        rows.append(
            "| " + " | ".join(values) + " |"
        )

    return "\n".join(
        [header, separator, *rows]
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = (
    PROJECT_ROOT / "configs" / "agelens_config.json"
)
DECISION_LOG_PATH = (
    PROJECT_ROOT / "docs" / "governance" / "Decision_Log.md"
)
PROTOCOL_PATH = (
    PROJECT_ROOT
    / "docs"
    / "methodology"
    / "Mortality_Analysis_Protocol.md"
)

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuration not found: {CONFIG_PATH}"
    )

if not DECISION_LOG_PATH.exists():
    raise FileNotFoundError(
        f"Decision Log not found: {DECISION_LOG_PATH}"
    )

CONFIG = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

PROCESSED_ROOT = (
    PROJECT_ROOT / CONFIG["paths"]["processed_data"]
)
INTERIM_ROOT = (
    PROJECT_ROOT / CONFIG["paths"]["interim_data"]
)
TABLES_ROOT = PROJECT_ROOT / CONFIG["paths"]["tables"]
LOGS_ROOT = PROJECT_ROOT / CONFIG["paths"]["logs"]

for path in [
    PROCESSED_ROOT,
    TABLES_ROOT,
    LOGS_ROOT,
    PROTOCOL_PATH.parent,
]:
    path.mkdir(parents=True, exist_ok=True)

CANONICAL_PATH = (
    PROCESSED_ROOT
    / "agelens_v1_canonical_complete_case.parquet"
)
PREPROCESSED_PATH = (
    INTERIM_ROOT
    / "nhanes_2015_2018_preprocessed_diagnostic.parquet"
)
MORTALITY_PATH = (
    INTERIM_ROOT
    / "nhanes_2015_2018_mortality_2019_public.parquet"
)

required_inputs = [
    CANONICAL_PATH,
    PREPROCESSED_PATH,
    MORTALITY_PATH,
]

missing_inputs = [
    path
    for path in required_inputs
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        f"Required mortality authorization inputs are missing: "
        f"{missing_inputs}"
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Canonical input: {CANONICAL_PATH}")
print(f"Demographic source: {PREPROCESSED_PATH}")
print(f"Mortality input: {MORTALITY_PATH}")
print(f"Decision Log: {DECISION_LOG_PATH}")
print(f"Mortality protocol: {PROTOCOL_PATH}")


Project root: <PROJECT_ROOT>
Canonical input: <PROJECT_ROOT>\data\processed\agelens_v1_canonical_complete_case.parquet
Demographic source: <PROJECT_ROOT>\data\interim\nhanes_2015_2018_preprocessed_diagnostic.parquet
Mortality input: <PROJECT_ROOT>\data\interim\nhanes_2015_2018_mortality_2019_public.parquet
Decision Log: <PROJECT_ROOT>\docs\governance\Decision_Log.md
Mortality protocol: <PROJECT_ROOT>\docs\methodology\Mortality_Analysis_Protocol.md


## 1. Verify release and governance prerequisites


In [6]:
release_gates = CONFIG.get(
    "release_gates",
    {},
)

required_true_gates = [
    "governance_core_gaps_resolved",
    "validation_completed",
    "canonical_outputs_regenerated",
    "canonical_regression_checks_passed",
    "cross_sectional_v1_results_allowed",
]

for gate in required_true_gates:
    if not release_gates.get(gate, False):
        raise RuntimeError(
            f"Required release gate is not open: {gate}"
        )

if release_gates.get(
    "mortality_analysis_authorized",
    False,
):
    raise RuntimeError(
        "Mortality analysis is already authorized in the "
        "configuration. Do not apply D-015 twice."
    )

if CONFIG["governance"].get(
    "open_core_evidence_gaps"
) != []:
    raise RuntimeError(
        "Open Core Evidence Gaps remain."
    )

if CONFIG["formula"].get(
    "primary_variant"
) != "supplement":
    raise RuntimeError(
        "The canonical formula is not Supplement."
    )

decision_text = DECISION_LOG_PATH.read_text(
    encoding="utf-8"
)

if extract_document_version(decision_text) != "1.7":
    raise RuntimeError(
        "Decision Log must be version 1.7 before D-015."
    )

if f"### {DECISION_ID}" in decision_text:
    raise RuntimeError(
        f"{DECISION_ID} already exists in the Decision Log."
    )

if PROTOCOL_PATH.exists():
    raise RuntimeError(
        "Mortality_Analysis_Protocol.md already exists. "
        "Review it before rerunning this authorization."
    )

print("✅ Mortality authorization prerequisites verified.")


RuntimeError: Mortality analysis is already authorized in the configuration. Do not apply D-015 twice.

## 2. Read and normalize the linked mortality file

Only the separately ingested public-use linked mortality file is used. No mortality fields are imported from the cross-sectional canonical output.


`RIAGENDR` and `RIDRETH3` were not included in the minimal canonical export produced by notebook 08. Notebook 09 restores these governed covariates from the preprocessed participant file using a validated one-to-one merge on `NHANES_CYCLE + SEQN`. Canonical Phenotypic Age values are not recalculated or altered.


In [ ]:
canonical = pd.read_parquet(
    CANONICAL_PATH
)
preprocessed = pd.read_parquet(
    PREPROCESSED_PATH
)
mortality = pd.read_parquet(
    MORTALITY_PATH
)

if (
    "NHANES_CYCLE" not in mortality.columns
    and "cycle" in mortality.columns
):
    mortality = mortality.rename(
        columns={"cycle": "NHANES_CYCLE"}
    )

canonical_required = {
    "SEQN",
    "NHANES_CYCLE",
    "chronological_age_years",
    "age_topcoded",
    "WTSAF4YR",
    "SDMVSTRA",
    "SDMVPSU",
    "phenotypic_age_years",
    "sensitivity_erratum_phenoage_years",
    "sensitivity_creatinine_plus_0_11_phenoage_years",
    "sensitivity_creatinine_plus_0_17_phenoage_years",
    "sensitivity_creatinine_plus_0_23_phenoage_years",
}

missing_canonical = sorted(
    canonical_required - set(canonical.columns)
)

if missing_canonical:
    raise ValueError(
        "Canonical participant file is missing required "
        f"columns: {missing_canonical}"
    )

demographic_required = {
    "SEQN",
    "NHANES_CYCLE",
    "RIAGENDR",
    "RIDRETH3",
}

missing_demographics = sorted(
    demographic_required - set(preprocessed.columns)
)

if missing_demographics:
    raise ValueError(
        "Preprocessed demographic source is missing required "
        f"columns: {missing_demographics}"
    )

mortality_required = {
    "SEQN",
    "NHANES_CYCLE",
    "ELIGSTAT",
    "MORTSTAT",
    "PERMTH_EXM",
}

missing_mortality = sorted(
    mortality_required - set(mortality.columns)
)

if missing_mortality:
    raise ValueError(
        "Mortality file is missing required columns: "
        f"{missing_mortality}"
    )

for frame_name, frame in {
    "canonical": canonical,
    "preprocessed": preprocessed,
    "mortality": mortality,
}.items():
    frame["SEQN"] = pd.to_numeric(
        frame["SEQN"],
        errors="raise",
    ).astype("int64")

    if frame.duplicated(
        ["NHANES_CYCLE", "SEQN"]
    ).any():
        raise RuntimeError(
            f"Duplicate cycle + SEQN rows exist in "
            f"{frame_name} data."
        )

demographics = preprocessed.loc[
    :,
    [
        "SEQN",
        "NHANES_CYCLE",
        "RIAGENDR",
        "RIDRETH3",
    ],
].copy()

canonical = canonical.merge(
    demographics,
    on=["SEQN", "NHANES_CYCLE"],
    how="left",
    validate="one_to_one",
)

if canonical[
    ["RIAGENDR", "RIDRETH3"]
].isna().any().any():
    missing_demo_rows = canonical.loc[
        canonical[
            ["RIAGENDR", "RIDRETH3"]
        ].isna().any(axis=1),
        ["NHANES_CYCLE", "SEQN"],
    ]

    raise RuntimeError(
        "Canonical participants are missing governed "
        "demographic covariates after the one-to-one merge: "
        f"{len(missing_demo_rows)}"
    )

mortality_columns = [
    column
    for column in [
        "SEQN",
        "NHANES_CYCLE",
        "ELIGSTAT",
        "MORTSTAT",
        "UCOD_LEADING",
        "DIABETES",
        "HYPERTEN",
        "PERMTH_INT",
        "PERMTH_EXM",
    ]
    if column in mortality.columns
]

merged = canonical.merge(
    mortality.loc[:, mortality_columns],
    on=["SEQN", "NHANES_CYCLE"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

if not merged["_merge"].eq("both").all():
    missing_linkage_rows = merged.loc[
        ~merged["_merge"].eq("both"),
        ["NHANES_CYCLE", "SEQN"],
    ]

    raise RuntimeError(
        "Canonical participants are missing mortality linkage "
        f"records: {len(missing_linkage_rows)}"
    )

merged = merged.drop(columns="_merge")

for column in [
    "ELIGSTAT",
    "MORTSTAT",
    "PERMTH_EXM",
]:
    merged[column] = pd.to_numeric(
        merged[column],
        errors="coerce",
    )

print(
    "Demographic covariates restored from the governed "
    "preprocessed file:"
)
print(
    f"  RIAGENDR non-missing: "
    f"{int(canonical['RIAGENDR'].notna().sum()):,}/"
    f"{len(canonical):,}"
)
print(
    f"  RIDRETH3 non-missing: "
    f"{int(canonical['RIDRETH3'].notna().sum()):,}/"
    f"{len(canonical):,}"
)
print(
    f"Canonical rows joined to mortality records: "
    f"{len(merged):,}"
)


Demographic covariates restored from the governed preprocessed file:
  RIAGENDR non-missing: 5,223/5,223
  RIDRETH3 non-missing: 5,223/5,223
Canonical rows joined to mortality records: 5,223


## 3. Define the governed mortality cohort


Public-use linked-mortality columns may be represented with pandas nullable dtypes. Linkage-ineligible records can therefore produce `pd.NA` during equality comparisons. All cohort and event masks are explicitly converted to two-valued Boolean arrays, with unresolved values treated as ineligible rather than as events.


In [ ]:
# Public-use mortality variables may use pandas nullable
# dtypes. Linkage-ineligible records can therefore produce
# `pd.NA` in equality comparisons. Every cohort mask is made
# explicitly two-valued before it is used for indexing or
# passed to NumPy.
merged["mortality_age_eligible"] = (
    merged["chronological_age_years"]
    .ge(20)
    .fillna(False)
    .astype(bool)
)
merged["mortality_linkage_eligible"] = (
    merged["ELIGSTAT"]
    .eq(1)
    .fillna(False)
    .astype(bool)
)
merged["mortality_outcome_observed"] = (
    merged["MORTSTAT"]
    .isin([0, 1])
    .fillna(False)
    .astype(bool)
)
merged["mortality_followup_valid"] = (
    (
        merged["PERMTH_EXM"].notna()
        & merged["PERMTH_EXM"].gt(0).fillna(False)
    )
    .fillna(False)
    .astype(bool)
)

merged["mortality_cohort_eligible"] = (
    (
        merged["mortality_age_eligible"]
        & merged["mortality_linkage_eligible"]
        & merged["mortality_outcome_observed"]
        & merged["mortality_followup_valid"]
    )
    .fillna(False)
    .astype(bool)
)

mortality_event_mask = (
    merged["MORTSTAT"]
    .eq(1)
    .fillna(False)
    .astype(bool)
)

merged["mortality_event"] = np.where(
    mortality_event_mask.to_numpy(dtype=bool),
    1,
    0,
).astype("int8")

merged["followup_months"] = merged[
    "PERMTH_EXM"
]
merged["followup_years"] = (
    merged["followup_months"] / 12.0
)

cohort = merged.loc[
    merged["mortality_cohort_eligible"]
].copy()

if cohort.empty:
    raise RuntimeError(
        "The governed mortality cohort is empty."
    )

if cohort[
    "chronological_age_years"
].lt(20).any():
    raise RuntimeError(
        "The governed mortality cohort contains participants "
        "younger than 20."
    )

if not cohort["mortality_event"].isin(
    [0, 1]
).all():
    raise RuntimeError(
        "Mortality event contains values outside 0/1."
    )

if cohort["followup_months"].le(0).any():
    raise RuntimeError(
        "The mortality cohort contains non-positive follow-up."
    )

exclusion_audit = pd.DataFrame(
    [
        {
            "step": "Canonical complete-case participants",
            "n": int(len(merged)),
        },
        {
            "step": "Age 20 years or older",
            "n": int(
                merged[
                    "mortality_age_eligible"
                ].sum()
            ),
        },
        {
            "step": "NCHS linkage eligible",
            "n": int(
                (
                    merged[
                        "mortality_age_eligible"
                    ]
                    & merged[
                        "mortality_linkage_eligible"
                    ]
                ).sum()
            ),
        },
        {
            "step": "Observed all-cause mortality outcome",
            "n": int(
                (
                    merged[
                        "mortality_age_eligible"
                    ]
                    & merged[
                        "mortality_linkage_eligible"
                    ]
                    & merged[
                        "mortality_outcome_observed"
                    ]
                ).sum()
            ),
        },
        {
            "step": "Positive MEC-based follow-up",
            "n": int(len(cohort)),
        },
    ]
)

cycle_audit = (
    cohort.groupby(
        "NHANES_CYCLE",
        observed=True,
    )
    .agg(
        n=("SEQN", "size"),
        deaths=("mortality_event", "sum"),
        weighted_population_sum=(
            "WTSAF4YR",
            "sum",
        ),
        followup_months_min=(
            "followup_months",
            "min",
        ),
        followup_months_median=(
            "followup_months",
            "median",
        ),
        followup_months_max=(
            "followup_months",
            "max",
        ),
        person_years=(
            "followup_years",
            "sum",
        ),
        age_topcoded_n=(
            "age_topcoded",
            "sum",
        ),
    )
    .reset_index()
)

if cycle_audit["deaths"].le(0).any():
    raise RuntimeError(
        "At least one cycle has no observed deaths."
    )

display(exclusion_audit)
display(cycle_audit.round(6))


,step,n
0,Canonical complete-case participants,5223
1,Age 20 years or older,4367
2,NCHS linkage eligible,4351
3,Observed all-cause mortality outcome,4351
4,Positive MEC-based follow-up,4350


,NHANES_CYCLE,n,deaths,weighted_population_sum,followup_months_min,followup_months_median,followup_months_max,person_years,age_topcoded_n
0,2015_2016,2178,83,1.126293e+08,1,47.0,61,8485.25,121
1,2017_2018,2172,44,1.140579e+08,1,24.0,37,4263.5,150


## 4. Create cycle-specific weighted Phenotypic Age acceleration

Residualization is performed separately within each cycle using `WTSAF4YR`. This prevents the cross-cycle mean shift from being absorbed into the acceleration exposure.


In [ ]:
def weighted_linear_residual(
    frame: pd.DataFrame,
    *,
    outcome: str,
    predictor: str,
    weight: str,
) -> tuple[pd.Series, dict[str, float]]:
    valid = frame.loc[
        :,
        [outcome, predictor, weight],
    ].dropna()

    valid = valid.loc[
        valid[weight].gt(0)
    ]

    x = valid[predictor].to_numpy(dtype=float)
    y = valid[outcome].to_numpy(dtype=float)
    w = valid[weight].to_numpy(dtype=float)

    x_mean = float(
        np.average(x, weights=w)
    )
    y_mean = float(
        np.average(y, weights=w)
    )

    denominator = float(
        np.sum(
            w * np.square(x - x_mean)
        )
    )

    if denominator <= 0:
        raise RuntimeError(
            "Weighted age variance is non-positive."
        )

    slope = float(
        np.sum(
            w
            * (x - x_mean)
            * (y - y_mean)
        )
        / denominator
    )
    intercept = y_mean - slope * x_mean

    fitted = (
        intercept
        + slope
        * frame[predictor].to_numpy(dtype=float)
    )
    residual = (
        frame[outcome].to_numpy(dtype=float)
        - fitted
    )

    return (
        pd.Series(
            residual,
            index=frame.index,
        ),
        {
            "intercept": intercept,
            "slope": slope,
            "weighted_age_mean": x_mean,
            "weighted_phenoage_mean": y_mean,
        },
    )


acceleration_audit_records = []

exposure_specs = {
    "canonical": "phenotypic_age_years",
    "erratum": (
        "sensitivity_erratum_phenoage_years"
    ),
    "creatinine_plus_0_11": (
        "sensitivity_creatinine_plus_0_11_phenoage_years"
    ),
    "creatinine_plus_0_17": (
        "sensitivity_creatinine_plus_0_17_phenoage_years"
    ),
    "creatinine_plus_0_23": (
        "sensitivity_creatinine_plus_0_23_phenoage_years"
    ),
}

for exposure_name, outcome_column in exposure_specs.items():
    output_column = (
        "phenoage_acceleration_years"
        if exposure_name == "canonical"
        else (
            f"sensitivity_{exposure_name}_"
            "acceleration_years"
        )
    )

    cohort[output_column] = np.nan

    for cycle, cycle_index in cohort.groupby(
        "NHANES_CYCLE",
        observed=True,
    ).groups.items():
        cycle_frame = cohort.loc[
            cycle_index
        ]

        residual, audit = weighted_linear_residual(
            cycle_frame,
            outcome=outcome_column,
            predictor="chronological_age_years",
            weight="WTSAF4YR",
        )

        cohort.loc[
            cycle_index,
            output_column,
        ] = residual

        weighted_residual_mean = float(
            np.average(
                residual,
                weights=cycle_frame[
                    "WTSAF4YR"
                ],
            )
        )

        acceleration_audit_records.append(
            {
                "exposure": exposure_name,
                "cycle": cycle,
                "n": int(len(cycle_frame)),
                **audit,
                "weighted_residual_mean": (
                    weighted_residual_mean
                ),
            }
        )

acceleration_audit = pd.DataFrame(
    acceleration_audit_records
)

if acceleration_audit[
    "weighted_residual_mean"
].abs().max() > 1e-10:
    raise RuntimeError(
        "A cycle-specific weighted acceleration residual "
        "does not have weighted mean zero."
    )

canonical_acceleration_sd = float(
    np.sqrt(
        np.average(
            np.square(
                cohort[
                    "phenoage_acceleration_years"
                ]
                - np.average(
                    cohort[
                        "phenoage_acceleration_years"
                    ],
                    weights=cohort["WTSAF4YR"],
                )
            ),
            weights=cohort["WTSAF4YR"],
        )
    )
)

if canonical_acceleration_sd <= 0:
    raise RuntimeError(
        "Canonical acceleration SD is non-positive."
    )

cohort[
    "phenoage_acceleration_per_5_years"
] = (
    cohort[
        "phenoage_acceleration_years"
    ]
    / 5.0
)
cohort[
    "phenoage_acceleration_per_sd"
] = (
    cohort[
        "phenoage_acceleration_years"
    ]
    / canonical_acceleration_sd
)

cohort["early_death_within_12_months"] = (
    (
        cohort["mortality_event"].eq(1)
        & cohort[
            "followup_months"
        ].le(12).fillna(False)
    )
    .fillna(False)
    .astype(bool)
)

age_topcoded_mask = (
    cohort["age_topcoded"]
    .fillna(False)
    .astype(bool)
)

cohort["no_topcode_sensitivity_eligible"] = (
    ~age_topcoded_mask
)
cohort["early_death_exclusion_eligible"] = (
    ~cohort[
        "early_death_within_12_months"
    ].astype(bool)
)

cohort["pooled_stratum"] = (
    cohort["NHANES_CYCLE"].astype(str)
    + "__"
    + cohort["SDMVSTRA"].astype(str)
)
cohort["pooled_psu"] = (
    cohort["NHANES_CYCLE"].astype(str)
    + "__"
    + cohort["SDMVSTRA"].astype(str)
    + "__"
    + cohort["SDMVPSU"].astype(str)
)

display(acceleration_audit.round(10))
print(
    "Canonical weighted acceleration SD: "
    f"{canonical_acceleration_sd:.6f} years"
)


,exposure,cycle,n,intercept,slope,weighted_age_mean,weighted_phenoage_mean,weighted_residual_mean
0,canonical,2015_2016,2178,-4.113863,1.059669,47.734013,46.468398,-0.0
1,canonical,2017_2018,2172,-3.895485,1.067666,48.085222,47.443492,-0.0
2,erratum,2015_2016,2178,-1.756702,1.042499,47.734013,48.005977,-0.0
3,erratum,2017_2018,2172,-1.541864,1.050367,48.085222,48.965271,-0.0
4,creatinine_plus_0_11,2015_2016,2178,-3.089319,1.059669,47.734013,47.492942,-0.0
5,creatinine_plus_0_11,2017_2018,2172,-2.870942,1.067666,48.085222,48.468035,-0.0
6,creatinine_plus_0_17,2015_2016,2178,-2.530476,1.059669,47.734013,48.051784,0.0
7,creatinine_plus_0_17,2017_2018,2172,-2.312099,1.067666,48.085222,49.026878,-0.0
8,creatinine_plus_0_23,2015_2016,2178,-1.971634,1.059669,47.734013,48.610627,-0.0
9,creatinine_plus_0_23,2017_2018,2172,-1.753257,1.067666,48.085222,49.585720,-0.0


Canonical weighted acceleration SD: 7.373384 years


## 5. Construct D-015 and the Mortality Analysis Protocol


In [ ]:
decision_text = replace_document_field(
    decision_text,
    "Version",
    "1.8",
)
decision_text = replace_document_field(
    decision_text,
    "Last Updated",
    AUTHORIZATION_DATE,
)
decision_text = add_revision_row(
    decision_text,
    (
        "| 1.8 | 2026-07-22 | Approved D-015: "
        "mortality cohort, all-cause outcome, MEC-based "
        "follow-up, acceleration exposure, survey Cox model "
        "scope, required sensitivities, and mortality release "
        "gate. |"
    ),
)

decision_entry = f"""
{AUTHORIZATION_MARKER}

---

### D-015

| Field | Value |
| --- | --- |
| Title | AgeLens V1 linked-mortality cohort and analysis authorization |
| Description | Authorize all-cause mortality analysis among canonical harmonized complete cases aged 20 years or older with `ELIGSTAT == 1`, observed `MORTSTAT`, and positive `PERMTH_EXM`. Use MEC examination as time origin. Use cycle-specific `WTSAF4YR`-weighted residuals of canonical Supplement Phenotypic Age on chronological age as the primary exposure. Fit survey-weighted Cox models with cycle-unique strata and PSUs. |
| Related Research Question | RQ4, RQ5 |
| Supporting Evidence | Canonical V1 rebuild passed 29/29 regression checks; linked mortality ingestion remained separate; authorization cohort contains {len(cohort):,} participants and {int(cohort['mortality_event'].sum()):,} deaths. |
| Evidence Level | Direct governed data audit plus official linked-mortality variable definitions already incorporated in the project |
| Confidence Rating | High for cohort construction and all-cause outcome definition; model estimates remain unvalidated until notebook 10 |
| Reviewer | Project owner |
| Status | Approved |
| Date | 2026-07-22 |
| Related Assumptions | MEC examination is the governed baseline; public-use follow-up months are used as released |
| Related Evidence Gaps | None blocking model execution |
| Notes | Primary HR is reported per 5-year higher acceleration. Adjusted models include chronological age, sex, race/ethnicity, and cycle. Required sensitivities are no-topcode, Erratum constants, three D-012 creatinine shifts, and exclusion of deaths within 12 months. Cause-specific mortality is not authorized. Mortality results remain non-reportable until notebook 10 passes its validation and release gate. |
"""

decision_text = (
    decision_text.rstrip()
    + "\n\n"
    + decision_entry.strip()
    + "\n"
)

protocol = f"""# AgeLens Mortality Analysis Protocol

## Document Control

| Field | Value |
| --- | --- |
| Document | Mortality Analysis Protocol |
| Version | 1.0 |
| Status | Approved for model execution |
| Last Updated | 2026-07-22 |
| Governing Decision | D-015 |

## 1. Scope

This protocol governs the first AgeLens V1 linked-mortality analysis. It does not alter the released cross-sectional canonical outputs.

## 2. Data Sources

- Canonical participant file: `data/processed/agelens_v1_canonical_complete_case.parquet`
- Public-use linked mortality file: `data/interim/nhanes_2015_2018_mortality_2019_public.parquet`
- NHANES cycles: 2015–2016 and 2017–2018
- Mortality follow-up release: public-use follow-up through 2019

## 3. Primary Cohort

Participants must satisfy all of the following:

1. Canonical harmonized complete-case membership.
2. Chronological age at least 20 years.
3. Positive pooled fasting weight `WTSAF4YR`.
4. `ELIGSTAT == 1`.
5. `MORTSTAT` observed as 0 or 1.
6. Positive `PERMTH_EXM`.

No imputation is permitted.

### Cohort flow

{dataframe_to_markdown(exclusion_audit)}

### Cycle audit

{dataframe_to_markdown(cycle_audit.round(6))}

## 4. Outcome and Time Scale

- Primary outcome: all-cause mortality.
- Event indicator: `MORTSTAT == 1`.
- Time origin: MEC examination.
- Follow-up variable: `PERMTH_EXM`.
- Analysis time unit: months; hazard ratios are invariant to rescaling time to years.
- Survivors remain censored at their released public-use follow-up duration.

Cause-specific mortality is not authorized in V1.

## 5. Primary Exposure

Canonical Phenotypic Age acceleration is the cycle-specific survey-weighted residual from:

`canonical Supplement Phenotypic Age ~ chronological age`

Residualization uses `WTSAF4YR` separately within each cycle.

Primary reporting scale:

- hazard ratio per 5-year higher acceleration.

Secondary reporting scale:

- hazard ratio per one weighted SD higher acceleration.

## 6. Survey Design

- Weight: `WTSAF4YR`
- Strata: cycle-unique combination of `NHANES_CYCLE` and `SDMVSTRA`
- PSU: cycle-unique combination of cycle, stratum, and `SDMVPSU`
- Planned estimator: R `survey::svycoxph`
- Cycles are pooled; cycle is included in adjusted models.

## 7. Models

### Model 0 — minimally adjusted

`Surv(PERMTH_EXM, mortality_event) ~ acceleration_per_5_years`

### Model 1 — primary adjusted model

`Surv(PERMTH_EXM, mortality_event) ~ acceleration_per_5_years + chronological_age_years + sex + race_ethnicity + cycle`

Categorical predictors must be treated as factors.

The analysis is prognostic and replicational, not causal.

## 8. Required Sensitivities

1. Exclude age-topcoded participants.
2. Replace canonical Supplement Phenotypic Age with the Erratum sensitivity.
3. Replace canonical creatinine scale with each D-012 sensitivity:
   - `+0.11 mg/dL`
   - `+0.17 mg/dL`
   - `+0.23 mg/dL`
4. Exclude deaths within the first 12 months.
5. Report cycle-specific event counts and descriptive follow-up.

## 9. Diagnostics

Notebook 10 must report:

- model convergence;
- coefficient, standard error, HR, and 95% CI;
- unweighted participant and event counts;
- weighted population sum;
- strata and PSU counts;
- time-interaction diagnostic for the primary acceleration exposure;
- comparison of Python cohort counts with R model input counts;
- independent verification that no cause-specific outcome was modeled.

A failed diagnostic blocks mortality-result release.

## 10. Release Gate

Model execution is authorized after notebook 09. Mortality results remain non-reportable until notebook 10:

1. completes every required model and sensitivity;
2. records no analysis errors;
3. passes cohort and survey-design reconciliation;
4. writes a model-validation report;
5. explicitly opens `mortality_results_allowed`.

{AUTHORIZATION_MARKER}
"""

if extract_document_version(decision_text) != "1.8":
    raise RuntimeError(
        "Updated Decision Log version is not 1.8."
    )

if f"### {DECISION_ID}" not in decision_text:
    raise RuntimeError(
        "D-015 was not constructed."
    )

if AUTHORIZATION_MARKER not in protocol:
    raise RuntimeError(
        "Mortality protocol marker is missing."
    )

print("✅ D-015 and mortality protocol constructed.")


✅ D-015 and mortality protocol constructed.


## 6. Write the authorized cohort and governance artifacts


In [ ]:
COHORT_PATH = (
    PROCESSED_ROOT
    / "agelens_v1_mortality_cohort_authorized.parquet"
)
EXCLUSION_AUDIT_PATH = (
    TABLES_ROOT
    / "09_mortality_cohort_flow.csv"
)
CYCLE_AUDIT_PATH = (
    TABLES_ROOT
    / "09_mortality_cycle_audit.csv"
)
ACCELERATION_AUDIT_PATH = (
    TABLES_ROOT
    / "09_acceleration_residualization_audit.csv"
)
AUTHORIZATION_CHECK_PATH = (
    TABLES_ROOT
    / "09_mortality_authorization_checks.csv"
)
WRITE_AUDIT_PATH = (
    TABLES_ROOT
    / "09_mortality_authorization_write_audit.csv"
)
METADATA_PATH = (
    LOGS_ROOT
    / "09_mortality_analysis_authorization_metadata.json"
)

cohort_output_columns = [
    "SEQN",
    "NHANES_CYCLE",
    "chronological_age_years",
    "age_topcoded",
    "RIAGENDR",
    "RIDRETH3",
    "WTSAF4YR",
    "SDMVSTRA",
    "SDMVPSU",
    "pooled_stratum",
    "pooled_psu",
    "phenotypic_age_years",
    "phenoage_acceleration_years",
    "phenoage_acceleration_per_5_years",
    "phenoage_acceleration_per_sd",
    "sensitivity_erratum_acceleration_years",
    "sensitivity_creatinine_plus_0_11_acceleration_years",
    "sensitivity_creatinine_plus_0_17_acceleration_years",
    "sensitivity_creatinine_plus_0_23_acceleration_years",
    "ELIGSTAT",
    "MORTSTAT",
    "mortality_event",
    "PERMTH_EXM",
    "followup_months",
    "followup_years",
    "early_death_within_12_months",
    "no_topcode_sensitivity_eligible",
    "early_death_exclusion_eligible",
]

optional_columns = [
    column
    for column in [
        "UCOD_LEADING",
        "DIABETES",
        "HYPERTEN",
        "PERMTH_INT",
    ]
    if column in cohort.columns
]

cohort_output = cohort.loc[
    :,
    cohort_output_columns + optional_columns,
].copy()

cohort_output["mortality_analysis_authorized"] = True
cohort_output["mortality_results_allowed"] = False
cohort_output["governing_decision"] = DECISION_ID

authorization_checks = pd.DataFrame(
    [
        {
            "check": "Canonical cross-sectional release open",
            "pass": bool(
                release_gates[
                    "cross_sectional_v1_results_allowed"
                ]
            ),
        },
        {
            "check": "No open Core Evidence Gaps",
            "pass": (
                CONFIG["governance"][
                    "open_core_evidence_gaps"
                ]
                == []
            ),
        },
        {
            "check": "All cohort participants are age 20+",
            "pass": bool(
                cohort[
                    "chronological_age_years"
                ].ge(20).all()
            ),
        },
        {
            "check": "All cohort participants are linkage eligible",
            "pass": bool(
                cohort["ELIGSTAT"].eq(1).all()
            ),
        },
        {
            "check": "All outcomes are observed binary values",
            "pass": bool(
                cohort["mortality_event"].isin(
                    [0, 1]
                ).all()
            ),
        },
        {
            "check": "All follow-up times are positive",
            "pass": bool(
                cohort["followup_months"].gt(0).all()
            ),
        },
        {
            "check": "Each cycle contains deaths",
            "pass": bool(
                cycle_audit["deaths"].gt(0).all()
            ),
        },
        {
            "check": "Acceleration residuals have weighted mean zero",
            "pass": bool(
                acceleration_audit[
                    "weighted_residual_mean"
                ].abs().max()
                <= 1e-10
            ),
        },
        {
            "check": "Cause-specific mortality remains unauthorized",
            "pass": True,
        },
        {
            "check": "Mortality results remain blocked",
            "pass": True,
        },
    ]
)

if not authorization_checks["pass"].all():
    display(
        authorization_checks.loc[
            ~authorization_checks["pass"]
        ]
    )
    raise RuntimeError(
        "Mortality authorization checks failed. "
        "No governance or configuration files were written."
    )

cohort_output.to_parquet(
    COHORT_PATH,
    index=False,
)
exclusion_audit.to_csv(
    EXCLUSION_AUDIT_PATH,
    index=False,
)
cycle_audit.to_csv(
    CYCLE_AUDIT_PATH,
    index=False,
)
acceleration_audit.to_csv(
    ACCELERATION_AUDIT_PATH,
    index=False,
)
authorization_checks.to_csv(
    AUTHORIZATION_CHECK_PATH,
    index=False,
)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

BACKUP_ROOT = (
    LOGS_ROOT
    / "mortality_authorization_backups"
    / timestamp
)
BACKUP_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

DECISION_BACKUP_PATH = (
    BACKUP_ROOT / "Decision_Log.md"
)
CONFIG_BACKUP_PATH = (
    BACKUP_ROOT / "agelens_config.json"
)

shutil.copy2(
    DECISION_LOG_PATH,
    DECISION_BACKUP_PATH,
)
shutil.copy2(
    CONFIG_PATH,
    CONFIG_BACKUP_PATH,
)

write_audit_records = [
    {
        "artifact": "decision_log",
        "path": str(
            DECISION_LOG_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "backup": str(
            DECISION_BACKUP_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256_before": sha256_file(
            DECISION_LOG_PATH
        ),
        "sha256_backup": sha256_file(
            DECISION_BACKUP_PATH
        ),
        "sha256_after": "",
    },
    {
        "artifact": "config",
        "path": str(
            CONFIG_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "backup": str(
            CONFIG_BACKUP_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256_before": sha256_file(
            CONFIG_PATH
        ),
        "sha256_backup": sha256_file(
            CONFIG_BACKUP_PATH
        ),
        "sha256_after": "",
    },
]

for record in write_audit_records:
    if (
        record["sha256_before"]
        != record["sha256_backup"]
    ):
        raise RuntimeError(
            f"Backup hash mismatch for {record['artifact']}."
        )

DECISION_LOG_PATH.write_text(
    decision_text,
    encoding="utf-8",
)
PROTOCOL_PATH.write_text(
    protocol,
    encoding="utf-8",
)

updated_config = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

updated_config["project"]["status"] = (
    "mortality_model_execution_authorized"
)

updated_config["governance"][
    "mortality_analysis_decision"
] = DECISION_ID

updated_config["mortality_analysis"] = {
    "decision": DECISION_ID,
    "status": "authorized_for_model_execution",
    "primary_outcome": "all_cause_mortality",
    "event_column": "mortality_event",
    "time_origin": "MEC examination",
    "followup_column": "PERMTH_EXM",
    "minimum_age_years": 20,
    "linkage_eligibility": "ELIGSTAT == 1",
    "primary_exposure": (
        "cycle-specific WTSAF4YR-weighted "
        "Supplement Phenotypic Age acceleration"
    ),
    "primary_exposure_scale_years": 5,
    "secondary_exposure_scale": (
        "one weighted standard deviation"
    ),
    "primary_model_covariates": [
        "chronological_age_years",
        "RIAGENDR",
        "RIDRETH3",
        "NHANES_CYCLE",
    ],
    "survey_weight": "WTSAF4YR",
    "survey_stratum": "pooled_stratum",
    "survey_psu": "pooled_psu",
    "planned_estimator": "survey::svycoxph",
    "required_sensitivities": [
        "no_topcode",
        "erratum_constants",
        "creatinine_plus_0_11_mg_dL",
        "creatinine_plus_0_17_mg_dL",
        "creatinine_plus_0_23_mg_dL",
        "exclude_deaths_within_12_months",
    ],
    "cause_specific_mortality_authorized": False,
    "cohort_file": str(
        COHORT_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "protocol_file": str(
        PROTOCOL_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
}

updated_config["release_gates"][
    "mortality_analysis_authorized"
] = True
updated_config["release_gates"][
    "mortality_cohort_defined"
] = True
updated_config["release_gates"][
    "mortality_models_completed"
] = False
updated_config["release_gates"][
    "mortality_model_validation_passed"
] = False
updated_config["release_gates"][
    "mortality_results_allowed"
] = False

updated_config["release_scope"][
    "mortality_analysis"
] = (
    "authorized_for_model_execution"
)
updated_config["release_scope"][
    "mortality_results"
] = "not_authorized"

CONFIG_PATH.write_text(
    json.dumps(
        updated_config,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

for record in write_audit_records:
    record["sha256_after"] = sha256_file(
        PROJECT_ROOT / record["path"]
    )

    if (
        record["sha256_after"]
        == record["sha256_before"]
    ):
        raise RuntimeError(
            f"{record['artifact']} did not change."
        )

write_audit = pd.DataFrame(
    write_audit_records
)
write_audit.to_csv(
    WRITE_AUDIT_PATH,
    index=False,
)

metadata = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook": (
        "09_mortality_analysis_authorization.ipynb"
    ),
    "decision": DECISION_ID,
    "cohort_rows": int(len(cohort_output)),
    "death_count": int(
        cohort_output["mortality_event"].sum()
    ),
    "canonical_acceleration_weighted_sd": (
        canonical_acceleration_sd
    ),
    "mortality_analysis_authorized": True,
    "mortality_results_allowed": False,
    "cause_specific_mortality_authorized": False,
    "next_notebook": (
        "10_mortality_survival_analysis.ipynb"
    ),
    "backup_root": str(
        BACKUP_ROOT.relative_to(
            PROJECT_ROOT
        )
    ),
}

METADATA_PATH.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

required_outputs = [
    COHORT_PATH,
    EXCLUSION_AUDIT_PATH,
    CYCLE_AUDIT_PATH,
    ACCELERATION_AUDIT_PATH,
    AUTHORIZATION_CHECK_PATH,
    WRITE_AUDIT_PATH,
    PROTOCOL_PATH,
    METADATA_PATH,
]

missing_outputs = [
    path
    for path in required_outputs
    if not path.exists()
]

if missing_outputs:
    raise RuntimeError(
        f"Expected mortality authorization outputs "
        f"were not created: {missing_outputs}"
    )

display(authorization_checks)
display(write_audit)

print("✅ Mortality cohort definition completed.")
print("✅ D-015 applied and model execution authorized.")
print("Mortality results remain non-reportable.")
print("Cause-specific mortality remains unauthorized.")
print(f"Authorized cohort rows: {len(cohort_output):,}")
print(
    f"Observed deaths: "
    f"{int(cohort_output['mortality_event'].sum()):,}"
)
print(f"Backup directory: {BACKUP_ROOT}")


,check,pass
0,Canonical cross-sectional release open,True
1,No open Core Evidence Gaps,True
2,All cohort participants are age 20+,True
3,All cohort participants are linkage eligible,True
4,All outcomes are observed binary values,True
5,All follow-up times are positive,True
6,Each cycle contains deaths,True
7,Acceleration residuals have weighted mean zero,True
8,Cause-specific mortality remains unauthorized,True
9,Mortality results remain blocked,True


,artifact,path,backup,sha256_before,sha256_backup,sha256_after
0,decision_log,docs\governance\Decision_Log.md,logs\mortality_authorization_backups\20260722T...,f93c540bc24ee3e5ed8d044f9f9df2a3db2b16c1e1c245...,f93c540bc24ee3e5ed8d044f9f9df2a3db2b16c1e1c245...,8f20c72180cc78e8b085298aad9c90b020daea1fe5711d...
1,config,configs\agelens_config.json,logs\mortality_authorization_backups\20260722T...,f75490901880e56a7e0ab0fd3d372b287d0ffd2d5fdb6b...,f75490901880e56a7e0ab0fd3d372b287d0ffd2d5fdb6b...,82fab97bf864c05c8ccd7c24bbc71bbbd2c485bc53afd8...


✅ Mortality cohort definition completed.
✅ D-015 applied and model execution authorized.
Mortality results remain non-reportable.
Cause-specific mortality remains unauthorized.
Authorized cohort rows: 4,350
Observed deaths: 127
Backup directory: <PROJECT_ROOT>\logs\mortality_authorization_backups\20260722T162516Z
